In [32]:
# Use kernal base (Python 3.13.5)
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel
import json
import os
import re
import time
import pprint
import pandas as pd 
from datetime import datetime

print(f"Loading .env from: {find_dotenv()}")
load_dotenv()  # Load environment variables from .env file

Loading .env from: /Users/prernaroy/Library/CloudStorage/OneDrive-UNSW/Year 2/disso/dissertation/.env


True

In [8]:
# === Configuration ===
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # Load the API key from the .env file
if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in your environment before running this notebook.")

client = OpenAI(api_key=OPENAI_API_KEY)

# Full Variable List Testing

Managed to get the output into JSON reliably for 4 variables. This will now be extended to more variables, including some text summary variables to see what the output is like. 

In [9]:
# Pydantic model for structured data extraction of Coronial findings and recommendations.
class VariablesToExtractExtended(BaseModel):
    file_number: str
    mental_disorder_flag: bool
    mental_disorder_labels: list[str]
    cognitive_disability_flag: bool
    cognitive_disability_labels: list[str]
    date_of_findings: str
    date_of_death: str
    age_at_death: int # Potential change to str for age estimates
    sex: str
    country_of_birth: str
    place_of_residence: str
    profession: str
    level_of_education: str
    comorbidities: list[str]
    medications: list[str]
    known_health_issues: list[str]
    place_of_death: str
    definitive_cause_of_death: str
    manner_of_death: str
    site_of_incident: str
    potential_causes_of_death: list[str]
    coroners_recommendations_summary: str
    custody_served: bool
    use_of_interpreters: bool
    preventable_death_flag: bool
    non_english_speaking_background_flag: bool
    circumstances_of_death_summary: str
    toxicology_results_summary: str
    domestic_violence_flag: bool

## Prompt Cells

In [10]:
instruction_text = "You are an expert at structured data extraction. You will be given unstructured text from a coronial findings and recommendations report and must convert it into the given structure."
# Compare to just asking straightforwardly without defining tone/goal -- to-do

In [11]:
system_content = "When coding for mental disorders and cognitive disabilities, recall that a mental health condition is a change in mood, thought, or behavior that causes distress or stops a person from doing daily tasks, and can start later in life. A cognitive disability limits intellectual function and adaptive skills, usually from an early age."
# Include a sentence here to specify what is a MHC and DD -- done 

In [12]:
user_text = """Extract the following medical and demographic information from the given coronial findings and recommmendations report. Only extract information present in the report. If a field is not present or unknown, return an empty string, an empty list for list fields, and False for boolean flags. Flags must be True if verifiably present in the report with supporting evidence such as doctor diagnoses, or stated clearly in the inquest report from a verifiable source. Outputs must be accurate and concise; do not provide additional commentary or explanation. Labels should be short, specific, and present in the report; do not paraphrase directly from the report. For summary variables, provide a summary of the key points if present, otherwise return an empty string. Do not provide potential speculative information when definitive information is in the report. Do not provide additional information beyond what is requested. Do not repeat information that is already in another variable. Use ICD-10 to accurately identify mental disorders and cognitive disabilities.

Extract the following fields: file_number, mental_disorder_flag, mental_disorder_labels, cognitive_disability_flag, cognitive_disability_labels, date_of_findings, date_of_death, age_at_death, sex, country_of_birth, place_of_residence, profession, level_of_education, comorbidities, medications, known_health_issues, place_of_death, definitive_cause_of_death, manner_of_death, site_of_incident, potential_causes_of_death, coroners_recommendations_summary, custody_served, use_of_interpreters, preventable_death_flag, non_english_speaking_background_flag, circumstances_of_death_summary, toxicology_results_summary, and domestic_violence_flag."""

# Include definition of each field in model prompt and what is the requirement of this field. have a very clear list of what to extract -- to-do
# Compare current prompt with no definitions/requirements and zero-shot with explanations for each field as well to see change in performance - done

# Research whether providing just sentence or the whole report (should be whole report) is sufficient for examples in few-shot 
# Few examples where I only give the sentence 
# Systematic approach on how to test this -- research 
# Reference Ida's dissertation - used whole discharge summary instead of just specific sections

In [13]:
file_url = "https://coroners.nsw.gov.au/documents/findings/2022/Inquest_into_the_death_of_Riley_Christopher_Shortland.pdf"
# file_url = "https://coroners.nsw.gov.au/documents/findings/2026/Inquest_into_the_death_of_BS.pdf"

Using this example, I see that there is an issue that needs to be addressed: the accurate definition of what is considered a mental disorder versus a developmental or cognitive disability. In different runs of this input, the output will sometimes only mark `cognitive_disability_flag` as `True` and `mental_disorder_flag` as `False`. However, in other runs, it considers ASD and ADHD to be mental disorders, while the only cognitive disability is "global developmental delay". 

This is something to think about, but for now I will focus on building the evaluation pipeline. For now I am creating my own evaluation pipeline, but it should be noted that there is a library called Promptfoo, which is now OpenAI's preferred way to conduct evaluations: https://developers.openai.com/cookbook/examples/evaluation/moving-from-openai-evals-to-promptfoo

## Model Run

In [14]:
client = OpenAI()

# Define the parsing request to extract structured data from a coronial finding and recommendations report.
start_time = time.time()
response = client.responses.parse(
    model="gpt-5.4-mini-2026-03-17",
    instructions = instruction_text,
    input=[
        {
            "role": "system",
            "content": system_content,
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": user_text
                },
                {
                    "type": "input_file",
                    "file_url": file_url,
                }
            ]
        }

    ],
    text_format=VariablesToExtractExtended,
)
latency = time.time() - start_time

report_variables_json = response.output_parsed.model_dump_json(indent=2)
# print(report_variables_json)

# Few-shot examples to specify what is MHC and disability 
# Check out Aaron's dissertation - he specifies in the prompt; use as an example of "do this and not that"

In [15]:
# Pydantic Model to Store outputs and runtime information
class InfoLog(BaseModel):
    response_id: str
    created_at: str
    completed_at: str
    latency_seconds: float
    instructions: str | None
    file_url: str
    system_content: str
    user_text: str
    metadata: dict
    model: str | None
    temperature: float
    top_p: float 
    max_output_tokens: int | None
    usage_input_tokens: int
    usage_output_tokens: int
    usage_total_tokens: int
    output: dict
    

In [16]:
log_data = InfoLog(
    response_id = response.id,
    created_at = str(datetime.fromtimestamp(response.created_at)),
    completed_at = str(datetime.fromtimestamp(response.completed_at)),
    latency_seconds = round(latency, 4),
    instructions = response.instructions,
    file_url = file_url,
    system_content = system_content,
    user_text = user_text,
    metadata = response.metadata,
    model = response.model,
    temperature = response.temperature,
    top_p = response.top_p,
    max_output_tokens = response.max_output_tokens,
    usage_input_tokens = response.usage.input_tokens,
    usage_output_tokens = response.usage.output_tokens,
    usage_total_tokens = response.usage.total_tokens,
    output = json.loads(response.output_text)
)

In [17]:
# log_data.model_dump_json

In [18]:
with open("audit_log.jsonl", "a") as f:
    f.write(log_data.model_dump_json() + "\n")

# Question -- I will record every prompt, but in terms of evaluation, do I need to do an evaluation after each run against the ground truth, even while I am fine-tuning the prompt?

# only evalate on the final prompt. record this process as the prompt development process -- manually evaluating 
# avoid the data leakage issue -- never ever use the record that is in the test set for evaluation during prompt development 

In [19]:
# log_data.instructions

In [20]:
# log_data.output

# Additive Prompting
This section replaces the `instructions=` + `system` role redundancy above with a single **top slot / user slot** model, which uses just the `instruction` parameters. Here we will build prompts up component-by-component for the additive ablation. It's designed so few-shot examples and chain-of-thought instructions can be added later as just another component. 

- **Top slot** = whatever you pass to `instructions=` (nothing else needed -- no separate `system` role message, as this is redundant per API documentation).
- **User slot** = the user turn: task instructions (if any) + the file.
- `channel` controls *where* the toggled-on components go: `"top_slot"`, `"user_slot"`, or `"none"` (a true minimal floor that ignores flags).

In [21]:
COMPONENTS = {
    "role_definition": ("You are an expert at structured data extraction. You will be given unstructured text from a coronial findings and recommendations report and must convert it into the given structure."),

    "mhc_dd_definitions": ("When coding for mental disorders and cognitive disabilities, recall that a mental health condition is a change in mood, thought, or behavior that causes distress or stops a person from doing daily tasks, and can start later in life. A cognitive disability limits intellectual function and adaptive skills, usually from an early age."),

    "icd10_instruction": ("Use ICD-10 to accurately identify mental disorders and cognitive disabilities."),

    "date_format_instruction": ("Report all dates exactly as they appear in the source document; do not reformat or infer missing date components."),

    "acronym_instruction": ("For diagnostic labels, use the full clinical term rather than an acronym (e.g. 'Attention-Deficit/Hyperactivity Disorder', not 'ADHD'), unless the source document uses only the acronym and gives no full form.")
}

In [22]:
TASK_CORE = ("""Extract the following medical and demographic information from the given coronial findings and recommendations report. Only extract information present in the report. If a field is not present or unknown, return an empty string, an empty list for list fields, and False for boolean flags. Flags must be True if verifiably present in the report with supporting evidence such as doctor diagnoses, or stated clearly in the inquest report from a verifiable source. Outputs must be accurate and concise; do not provide additional commentary or explanation. Labels should be short, specific, and present in the report; do not paraphrase directly from the report. For summary variables, provide a summary of the key points if present, otherwise return an empty string. Do not provide potential speculative information when definitive information is in the report. Do not provide additional information beyond what is requested. Do not repeat information that is already in another variable.

Extract the following fields: file_number, mental_disorder_flag, mental_disorder_labels, cognitive_disability_flag, cognitive_disability_labels, date_of_findings, date_of_death, age_at_death, sex, country_of_birth, place_of_residence, profession, level_of_education, comorbidities, medications, known_health_issues, place_of_death, definitive_cause_of_death, manner_of_death, site_of_incident, potential_causes_of_death, coroners_recommendations_summary, custody_served, use_of_interpreters, preventable_death_flag, non_english_speaking_background_flag, circumstances_of_death_summary, toxicology_results_summary, and domestic_violence_flag.""")

In [23]:
def compose(flags: dict, channel: str = "top_slot", include_task_core: bool = True) -> tuple[str, str]:
    """
    Turns a set of on/off flags into (top_slot_text, user_text).

    flags: e.g. {"role_definition": True, "icd10_instruction": True} -- keys match 
    
    COMPONENTS
    channel: "top_slot"  -> flagged component text goes into `instructions`, user_text = TASK_CORE (if included):
        "user_slot" -> flagged component text is prepended to user_text instead
        "none" -> flags ignored entirely; top slot is always empty
    
    include_task_core : whether to include the field-list/task instructions in the user turn at all.
        False reproduces the "schema + file only, no explicit task text" condition.
    """

    active_text = " ".join(text for name, text in COMPONENTS.items() if flags.get(name))
    task_core = TASK_CORE if include_task_core else ""

    if channel == "top_slot":
        return active_text, task_core
    elif channel == "user_slot":
        return "", (active_text + "\n\n" + task_core).strip()
    elif channel == "none":
        return "", task_core
    else:
        raise ValueError(f"Unknown channel: {channel}")


# Quick check --> top slot filled (role_definition + mhc_dd_definitions + icd10_instruction), user turn with NO task text, file only.
top_slot_text, user_text_check = compose(
    {"role_definition": True, "mhc_dd_definitions": True, "icd10_instruction": True},
    channel="top_slot",
    include_task_core=False,
)
print("TOP SLOT:\n", top_slot_text, "\n")
print("USER SLOT (should be empty string -> file-only user turn):", repr(user_text_check))

TOP SLOT:
 You are an expert at structured data extraction. You will be given unstructured text from a coronial findings and recommendations report and must convert it into the given structure. When coding for mental disorders and cognitive disabilities, recall that a mental health condition is a change in mood, thought, or behavior that causes distress or stops a person from doing daily tasks, and can start later in life. A cognitive disability limits intellectual function and adaptive skills, usually from an early age. Use ICD-10 to accurately identify mental disorders and cognitive disabilities. 

USER SLOT (should be empty string -> file-only user turn): ''


In [24]:
class PromptTrialLog(BaseModel):
    trial_id: str
    step: int | None = None    # position in the additive build-up, if applicable
    channel: str               # "top_slot" / "user_slot" / "none"
    include_task_core: bool    # was explicit task text included in the user turn?
    factors: dict              # e.g. {"role_definition": True, "icd10_instruction": False, ...}
    top_slot_text: str
    user_text: str
    file_url: str
    notes: str = ""
    response_id: str
    created_at: str
    completed_at: str
    latency_seconds: float
    model: str | None
    usage_input_tokens: int
    usage_output_tokens: int
    usage_total_tokens: int
    output: dict

In [25]:
def run_trial(flags: dict, channel: str = "top_slot", include_task_core: bool = True,
              file_url: str = file_url, model: str = "gpt-5.4-mini-2026-03-17",
              step: int | None = None, notes: str = "") -> dict:
    """Builds a request from flags+channel+include_task_core, calls the API once, logs the trial, returns the record."""

    top_slot_text, user_text = compose(flags, channel=channel, include_task_core=include_task_core)

    request_kwargs = {}
    if top_slot_text:
        request_kwargs["instructions"] = top_slot_text

    user_content = []
    if user_text:
        user_content.append({"type": "input_text", "text": user_text})
    user_content.append({"type": "input_file", "file_url": file_url})   # file is always present

    start_time = time.time()
    response = client.responses.parse(
        model=model,
        input=[{"role": "user", "content": user_content}],
        text_format=VariablesToExtractExtended,
        **request_kwargs,
    )
    latency = time.time() - start_time

    log_entry = PromptTrialLog(
        trial_id=f"{datetime.now().strftime('%Y%m%dT%H%M%SZ')}_{response.id[-8:]}",
        step=step,
        channel=channel,
        include_task_core=include_task_core,
        factors=flags,
        top_slot_text=top_slot_text,
        user_text=user_text,
        file_url=file_url,
        notes=notes,
        response_id=response.id,
        created_at=str(datetime.fromtimestamp(response.created_at)),
        completed_at=str(datetime.fromtimestamp(response.completed_at)),
        latency_seconds=round(latency, 4),
        model=response.model,
        usage_input_tokens=response.usage.input_tokens,
        usage_output_tokens=response.usage.output_tokens,
        usage_total_tokens=response.usage.total_tokens,
        output=json.loads(response.output_text),
    )

    with open("audit_log.jsonl", "a") as f:
        f.write(log_entry.model_dump_json() + "\n")

    return log_entry.model_dump()

In [27]:
# run_trial(flags={}, channel="none", include_task_core=False, step=None, notes="P0 floor")

run_trial(flags={}, channel="none", include_task_core=True,  step=None, notes="P1 task text only")

run_trial(flags={"role_definition": True}, channel="top_slot", include_task_core=True, step=2, notes="P2")

run_trial(flags={"role_definition": True, "mhc_dd_definitions": True}, channel="top_slot",
          include_task_core=True, step=3, notes="P3")

run_trial(flags={"role_definition": True, "mhc_dd_definitions": True, "icd10_instruction": True},
          channel="top_slot", include_task_core=True, step=4, notes="P4 ~ original prompt")

run_trial(flags={"role_definition": True, "mhc_dd_definitions": True, "icd10_instruction": True,
                  "date_format_instruction": True},
          channel="top_slot", include_task_core=True, step=5, notes="P5")

run_trial(flags={"role_definition": True, "mhc_dd_definitions": True, "icd10_instruction": True,
                  "date_format_instruction": True, "acronym_instruction": True},
          channel="top_slot", include_task_core=True, step=6, notes="P6 full prompt")

run_trial(flags={"role_definition": True, "mhc_dd_definitions": True, "icd10_instruction": True,
                  "date_format_instruction": True, "acronym_instruction": True},
          channel="user_slot", include_task_core=True, step=None, notes="P7 structural: slot placement")

run_trial(flags={"role_definition": True, "mhc_dd_definitions": True, "icd10_instruction": True,
                  "date_format_instruction": True, "acronym_instruction": True},
          channel="top_slot", include_task_core=False, step=None, notes="P8 structural: no task text")



{'trial_id': '20260819T084048Z_6f225274',
 'step': None,
 'channel': 'top_slot',
 'include_task_core': False,
 'factors': {'role_definition': True,
  'mhc_dd_definitions': True,
  'icd10_instruction': True,
  'date_format_instruction': True,
  'acronym_instruction': True},
 'top_slot_text': "You are an expert at structured data extraction. You will be given unstructured text from a coronial findings and recommendations report and must convert it into the given structure. When coding for mental disorders and cognitive disabilities, recall that a mental health condition is a change in mood, thought, or behavior that causes distress or stops a person from doing daily tasks, and can start later in life. A cognitive disability limits intellectual function and adaptive skills, usually from an early age. Use ICD-10 to accurately identify mental disorders and cognitive disabilities. Report all dates exactly as they appear in the source document; do not reformat or infer missing date components

In [31]:
log_df = pd.read_json("audit_log.jsonl", lines=True)
# log_df.head()

In [ ]:
# Only rows written by run_trial() have a "factors" column; expand it into one column per factor
factor_cols = pd.json_normalize(log_df["factors"]).add_prefix("factor__")
tracker_df = pd.concat(
    [log_df[["trial_id", "step", "channel", "model", "notes", "response_id", "latency_seconds"]], factor_cols],
    axis=1,
)
tracker_df.to_excel("zero_shot_prompt_tracker.xlsx", index=False)
tracker_df

,trial_id,step,channel,model,notes,response_id,latency_seconds,factor__role_definition,factor__mhc_dd_definitions,factor__icd10_instruction,factor__date_format_instruction,factor__acronym_instruction
0,20260819T083848Z_ac1577c7,NaN,none,gpt-5.4-mini-2026-03-17,P0 floor,resp_09876bf373d05019006a84deef62d0819a8bb1c9d...,9.1606,NaN,NaN,NaN,NaN,NaN
1,20260819T083953Z_49c15aa5,NaN,none,gpt-5.4-mini-2026-03-17,P1 task text only,resp_026d5c6a48154577006a84df32ca6487d0b426033...,7.2420,NaN,NaN,NaN,NaN,NaN
2,20260819T084002Z_df795e08,2.0,top_slot,gpt-5.4-mini-2026-03-17,P2,resp_0a2d0dc2c9c89ae5006a84df39bdd481988925dd7...,8.6902,True,NaN,NaN,NaN,NaN
3,20260819T084010Z_3d2eff50,3.0,top_slot,gpt-5.4-mini-2026-03-17,P3,resp_0a939384c08a7b60006a84df425e7881998b99017...,8.0455,True,True,NaN,NaN,NaN
4,20260819T084016Z_1b9ba11b,4.0,top_slot,gpt-5.4-mini-2026-03-17,P4 ~ original prompt,resp_023592e0e5f3e65a006a84df4a6a1887d082f8324...,6.1240,True,True,True,NaN,NaN
5,20260819T084023Z_dee5d928,5.0,top_slot,gpt-5.4-mini-2026-03-17,P5,resp_03e840ab2ba49843006a84df50d3d087d0880e7c3...,7.1934,True,True,True,True,NaN
6,20260819T084033Z_6c2ce7a4,6.0,top_slot,gpt-5.4-mini-2026-03-17,P6 full prompt,resp_045c716e613aa502006a84df57b7ec87d08ef97a4...,9.4289,True,True,True,True,True
7,20260819T084039Z_69b88775,NaN,user_slot,gpt-5.4-mini-2026-03-17,P7 structural: slot placement,resp_0c6cfbfe6c0428e8006a84df61286087d089adf91...,6.5005,True,True,True,True,True
8,20260819T084048Z_6f225274,NaN,top_slot,gpt-5.4-mini-2026-03-17,P8 structural: no task text,resp_04b38772e1862d57006a84df67ab8887d0a4bb1e5...,9.1798,True,True,True,True,True


In [33]:
log_df = pd.read_json("audit_log.jsonl", lines=True)

# Pull the "P0".."P8" label back out of the notes string set by the run_trial() calls above. Falls back to None for any run whose notes weren't written in that "P<n> ..." format.
log_df["prompt_id"] = log_df["notes"].str.extract(r"^(P\d+)")

# One column per extracted variable (file_number, mental_disorder_flag, ... all 29 fields)
output_df = pd.json_normalize(log_df["output"])

review_df = pd.concat(
    [log_df[["prompt_id", "trial_id", "channel", "include_task_core", "notes"]], output_df],
    axis=1,
).set_index(["prompt_id", "trial_id"]).sort_index()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

# Optional: export for scrolling/annotating outside the notebook
review_df.to_excel("human_review.xlsx")

review_df

,,channel,include_task_core,notes,file_number,mental_disorder_flag,mental_disorder_labels,cognitive_disability_flag,cognitive_disability_labels,date_of_findings,date_of_death,age_at_death,sex,country_of_birth,place_of_residence,profession,level_of_education,comorbidities,medications,known_health_issues,place_of_death,definitive_cause_of_death,manner_of_death,site_of_incident,potential_causes_of_death,coroners_recommendations_summary,custody_served,use_of_interpreters,preventable_death_flag,non_english_speaking_background_flag,circumstances_of_death_summary,toxicology_results_summary,domestic_violence_flag
prompt_id,trial_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
P0,20260819T083848Z_ac1577c7,none,False,P0 floor,2017/00335331,True,"[Autism Spectrum Disorder, Attention Deficit Hyperactivi...",True,"[Global Development Delay, moderate to severe global dev...",21 October 2022,5 November 2017,8,Male,,"Woy Woy, NSW, Australia",,,"[Autism Spectrum Disorder, Attention Deficit Hyperactivi...","[amphetamine, risperidone, citalopram]","[predominantly non-verbal, functioned at the development...","M1 Pacific Motorway near Cameron Park, NSW",Multiple injuries sustained when struck by a truck on th...,Accident,"M1 Pacific Motorway near Cameron Park, NSW",[Escaped from car during transport while in respite care...,Recommendations included requiring transport policies fo...,False,False,True,False,"Riley was in respite care with SNAP when, during the ret...","Riley's toxicology detected amphetamine, risperidone and...",False
P1,20260819T083953Z_49c15aa5,none,True,P1 task text only,2017/00335331,True,"[ADHD, Autism Spectrum Disorder]",True,[Global Development Delay],21 October 2022,5 November 2017,8,Male,,"Woy Woy, NSW",,,[],"[amphetamine, risperidone, citalopram]","[limited speech, non-verbal, developmental delay, behavi...",M1 Pacific Motorway near Cameron Park NSW,multiple injuries sustained when he was hit by a truck,Accident,M1 Motorway at Cameron Park,[multiple injuries],Recommend transport policies for SSRC/SRC providers; che...,False,False,True,False,Riley was in SNAP respite arranged by DCJ when he remove...,"Riley's toxicology detected amphetamine, risperidone and...",False
P2,20260819T084002Z_df795e08,top_slot,True,P2,2017/00335331,False,"[ADHD, Autism Spectrum Disorder]",True,[Global Development Delay],21 October 2022,5 November 2017,8,Male,,"Woy Woy, NSW",,,[],"[amphetamine, risperidone, citalopram]","[Global Development Delay, Autism Spectrum Disorder, ADH...",M1 Pacific Motorway near Cameron Park NSW,multiple injuries,Accident,M1 Motorway at Cameron Park,"[hit by a truck, escaped from car during respite transpo...",Recommended transport policies for SSRC/SRC providers; a...,False,False,True,False,Riley was in SNAP respite care under DCJ case management...,Toxicology found prescription drugs at therapeutic/ non-...,False
P3,20260819T084010Z_3d2eff50,top_slot,True,P3,2017/00335331,True,"[autism spectrum disorder, ADHD]",True,[global development delay],21 October 2022,5 November 2017,8,Male,,"Woy Woy, NSW",,,"[autism spectrum disorder, global development delay, att...","[amphetamine, risperidone, citalopram]","[non-verbal, limited speech and communication skills, re...",M1 Pacific Motorway near Cameron Park NSW,Multiple injuries sustained when struck by a truck on th...,Accidental,M1 Motorway at Cameron Park,"[multiple injuries, hit by a truck, escaped from car dur...",Recommended transport policies and checks for SSRC/SRC a...,False,False,True,False,Riley was in SNAP respite care during a return trip to h...,"Riley's toxicology detected amphetamine, risperidone and...",False
P4,20260819T084016Z_1b9ba11b,top_slot,True,P4 ~ original prompt,2017/00335331,True,"[ADHD, Autism Spectrum Disorder, Global Development Delay]",True,[Global Development Delay],21 October 2022,5 November 2017,8,Male,,"Woy Woy, NSW",,,[],"[amphetamine, risperidone, citalopram]","[limited speech, non-verbal, required close supervision,...",M1 Pacific Motorway near Cameron

## Evaluation Pipeline Setup

The goal of the evaluation pipeline is to:
1. Version prompts and record the model output for each prompt used (store the prompt and the model output for future reference).
2. Determine the formatting accuracy of the model output (it should be JSON). 
3. Measure the accuracy and relevancy of the model output (the value produced for each variable extracted). 

### Binary Variable Evaluation
Entity-matching performance will be evaluated using standard classification metrics such as accuracy, precision, recall, negative predictive value (NPV) and F1-score.

In [34]:
type(json.loads(response.output_text))

dict

In [ ]:
output_dict = json.loads(response.output_text)
output_dict
# Manual evaluation process as the main evaluation technique -- more accurate 

{'file_number': '2017/00335331',
 'mental_disorder_flag': False,
 'mental_disorder_labels': [],
 'cognitive_disability_flag': True,
 'cognitive_disability_labels': ['global development delay'],
 'date_of_findings': '21 October 2022',
 'date_of_death': '5 November 2017',
 'age_at_death': 8,
 'sex': 'Male',
 'country_of_birth': '',
 'place_of_residence': 'Woy Woy, NSW',
 'profession': '',
 'level_of_education': '',
 'comorbidities': ['autism spectrum disorder',
  'attention deficit hyperactivity disorder'],
 'medications': ['amphetamine', 'risperidone', 'citalopram'],
 'known_health_issues': ['moderate to severe global development delay',
  'ASD-3',
  'limited speech',
  'predominantly non-verbal',
  'required close supervision',
  'behaviours of concern when travelling in vehicles'],
 'place_of_death': 'M1 Pacific Motorway near Cameron Park NSW',
 'definitive_cause_of_death': 'multiple injuries sustained when hit by a truck',
 'manner_of_death': 'Accidental',
 'site_of_incident': 'M1 Mo

In [47]:
test1_df = pd.read_csv("test1.csv")
test1_dict = test1_df.to_dict(orient="records")

In [48]:
test1_dict = test1_dict[0]

In [ ]:
test1_dict
# Try address evaluation using Bertscore -- show next week 

{'file_number': '2017/00335331',
 'mental_disorder_flag': False,
 'mental_disorder_labels': nan,
 'cognitive_disability_flag': True,
 'cognitive_disability_labels': 'Autism Spetrum Disorder, Attention-Deficit/Hyperactivity Disorder, Global Developmental Delay',
 'date_of_findings': '21/10/2022',
 'date_of_death': '5/11/2017',
 'age_at_death': 8,
 'sex': 'Male',
 'country_of_birth': nan,
 'place_of_residence': 'Woy Woy, NSW',
 'profession': nan,
 'level_of_education': nan,
 'comorbidities': nan,
 'medications': 'amphetamine, risperidone, citalopram',
 'known_health_issues': 'Mostly non-verbal with limited speech and communication skills',
 'place_of_death': 'M1 Pacific Motorway near Cameron Park\nNSW',
 'definitive_cause_of_death': 'Multiple injuries sustained when he was\nhit by a truck',
 'manner_of_death': "Riley died while escaping from the care of Rachel\nMartin, the carer employed by SNAP Programs to care\nfor him while in respite care. It is likely that Riley\nremoved a car seat 

In [53]:
output_dict.keys() == test1_dict.keys()

True

In [54]:
# List of exact-match/binary/boolean variables
exact_match_variables = [
    "file_number",
    "date_of_findings",
    "sex"
]

boolean_variables = [
    "mental_disorder_flag",
    "cognitive_disability_flag",
    "preventable_death_flag",
    "use_of_interpreters",
    "non_english_speaking_background_flag",
    "custody_served"
]

In [58]:
def exact_match_eval(output_dict, gt_dict, variables_list):
    eval_dict = {}
    for var in variables_list:
        if output_dict[var] == gt_dict[var]:
            eval_dict[var] = 1
        else:
            eval_dict[var] = 0
    return eval_dict

In [60]:
eval_dict_bool = exact_match_eval(output_dict, test1_dict, boolean_variables)
eval_dict_bool

{'mental_disorder_flag': 1,
 'cognitive_disability_flag': 1,
 'preventable_death_flag': 1,
 'use_of_interpreters': 1,
 'non_english_speaking_background_flag': 1,
 'custody_served': 1}

In [62]:
eval_dict_exact_match = exact_match_eval(output_dict, test1_dict, exact_match_variables)
eval_dict_exact_match

{'file_number': 1, 'date_of_findings': 0, 'sex': 1}

In [63]:
output_dict["date_of_findings"]

'21 October 2022'

In [64]:
test1_dict["date_of_findings"]

'21/10/2022'

Important finding for dates: make sure that dates are recorded in a way that is consistent. Date of findings will always be exact. Date of death can vary depending on finding